In [ ]:
import cv2
import pandas as pd
import numpy as np
import psutil
from tqdm import tqdm
from typing import Literal

TRN_DIR = "./dl-lab-2-stuff-detection/yolo_dataset/yolo_dataset/train/images"
TST_DIR = "./dl-lab-2-stuff-detection/test_images/test_images"

MTRN = pd.read_csv('./trn_meta.csv', parse_dates=['datetime'])
MTST = pd.read_csv('./tst_meta.csv', parse_dates=['datetime'])

In [ ]:
def immedian(imgs: np.ndarray):
    return np.median(imgs, axis=(0,)).astype(np.uint8)

def imload(dir, fn) -> np.ndarray:
    return cv2.imread(f"{dir}/{fn}")  # type: ignore

def _check_mem(shape):
    expect_size = int(np.prod(shape))
    free_size = psutil.virtual_memory().available
    print(f"{expect_size / 1024**2:.02f} / {free_size / 1024**2:.02f} MB: "
          f"{'DECLINE' if expect_size > free_size else 'APPROVE'}", end='')
    if expect_size > free_size:
        raise RuntimeError

def locload(loc):
    split_idx = len(MTRN[MTRN['loc'] == loc]['fname'])
    fns = (tuple(MTRN[MTRN['loc'] == loc]['fname']) + tuple(MTST[MTST['loc'] == loc]['fname']))
    shape = len(fns), 720, 1280, 3
    _check_mem(shape)
    imgs = np.zeros(shape, dtype=np.uint8)
    pbar = tqdm(enumerate(fns), total=shape[0])
    for i, fn in pbar:
        pbar.set_description(f"loc={loc:02d}")
        dir = TRN_DIR if i < split_idx else TST_DIR
        imgs[i] = imload(dir, fn)
    return imgs

def job_loc_median(out_dir):
    for loc in range(40):
        med = immedian(locload(loc))
        outp = f"{out_dir}/median-{loc}.jpg"
        cv2.imwrite(outp, med)


In [ ]:
job_loc_median('./loc-median')